In [16]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("StudentsAnalysis").getOrCreate()


df = spark.read.csv("file:///D:/BDA 20/ABD-BDA-LabWork/datasets/0909 Lab/students.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)


root
 |-- gender: string (nullable = true)
 |-- race/ethnicity: string (nullable = true)
 |-- parental level of education: string (nullable = true)
 |-- lunch: string (nullable = true)
 |-- test preparation course: string (nullable = true)
 |-- math score: integer (nullable = true)
 |-- reading score: integer (nullable = true)
 |-- writing score: integer (nullable = true)

+------+--------------+---------------------------+------------+-----------------------+----------+-------------+-------------+
|gender|race/ethnicity|parental level of education|       lunch|test preparation course|math score|reading score|writing score|
+------+--------------+---------------------------+------------+-----------------------+----------+-------------+-------------+
|female|       group B|          bachelor's degree|    standard|                   none|        72|           72|           74|
|female|       group C|               some college|    standard|              completed|        69|           90

In [17]:
# ---------------------------------------------------------------------
# 1. Number of male and female students
# ---------------------------------------------------------------------
df.groupBy("gender").count().show()

# ---------------------------------------------------------------------
# 2. Different 'race/ethnicity' values
# ---------------------------------------------------------------------
df.select("race/ethnicity").distinct().show()

# ---------------------------------------------------------------------
# 3. Different 'parental level of education' values
# ---------------------------------------------------------------------
df.select("parental level of education").distinct().show(truncate=False)

+------+-----+
|gender|count|
+------+-----+
|female|  518|
|  male|  482|
+------+-----+

+--------------+
|race/ethnicity|
+--------------+
|       group B|
|       group C|
|       group D|
|       group A|
|       group E|
+--------------+

+---------------------------+
|parental level of education|
+---------------------------+
|some high school           |
|associate's degree         |
|high school                |
|bachelor's degree          |
|master's degree            |
|some college               |
+---------------------------+



In [18]:
# 4. Female students who scored > 79 in maths AND parental education = 'high school'

q4 = df.filter(
    (F.col("gender") == "female") &
    (F.col("math score") > 79) &
    (F.col("parental level of education") == "high school")
)
print(f"Female students, math > 79, parent education = high school: {q4.count()}")
q4.show(truncate=False)

Female students, math > 79, parent education = high school: 5
+------+--------------+---------------------------+--------+-----------------------+----------+-------------+-------------+
|gender|race/ethnicity|parental level of education|lunch   |test preparation course|math score|reading score|writing score|
+------+--------------+---------------------------+--------+-----------------------+----------+-------------+-------------+
|female|group B       |high school                |standard|none                   |87        |95           |86           |
|female|group E       |high school                |standard|none                   |99        |93           |90           |
|female|group D       |high school                |standard|completed              |88        |99           |100          |
|female|group B       |high school                |standard|none                   |81        |91           |89           |
|female|group C       |high school                |standard|none      

In [20]:
# 5. Average maths score: male vs female - which is higher?

avg_math = df.groupBy("gender").agg(F.round(F.avg("math score"), 2).alias("avg_math_score"))
avg_math.show()

top_math = avg_math.orderBy(F.desc("avg_math_score")).first()
print(f"Higher average maths score: {top_math['gender']} ({top_math['avg_math_score']})")

+------+--------------+
|gender|avg_math_score|
+------+--------------+
|female|         63.63|
|  male|         68.73|
+------+--------------+

Higher average maths score: male (68.73)


In [21]:
# 6. Average reading score of male and female students

avg_reading = df.groupBy("gender").agg(F.round(F.avg("reading score"), 2).alias("avg_reading_score"))
avg_reading.show()

+------+-----------------+
|gender|avg_reading_score|
+------+-----------------+
|female|            72.61|
|  male|            65.47|
+------+-----------------+



In [22]:
# 7. Does score depend on 'parental level of education'?
#    Compare average math/reading/writing score across education levels

score_by_education = df.groupBy("parental level of education").agg(
    F.round(F.avg("math score"), 2).alias("avg_math"),
    F.round(F.avg("reading score"), 2).alias("avg_reading"),
    F.round(F.avg("writing score"), 2).alias("avg_writing"),
    F.count("*").alias("num_students")
).orderBy(F.desc("avg_math"))
score_by_education.show(truncate=False)
# If avg scores clearly rise/fall across education levels (e.g. master's degree
# highest, some high school lowest), scores do depend on parental education.

+---------------------------+--------+-----------+-----------+------------+
|parental level of education|avg_math|avg_reading|avg_writing|num_students|
+---------------------------+--------+-----------+-----------+------------+
|master's degree            |69.75   |75.37      |75.68      |59          |
|bachelor's degree          |69.39   |73.0       |73.38      |118         |
|associate's degree         |67.88   |70.93      |69.9       |222         |
|some college               |67.13   |69.46      |68.84      |226         |
|some high school           |63.5    |66.94      |64.89      |179         |
|high school                |62.14   |64.7       |62.45      |196         |
+---------------------------+--------+-----------+-----------+------------+



In [23]:
# 8. Records where 'test preparation course' = 'none' and maths score > 70

q8 = df.filter(
    (F.col("test preparation course") == "none") &
    (F.col("math score") > 70)
)
print(f"Records with no test prep course and math score > 70: {q8.count()}")
q8.show(truncate=False)

spark.stop()

Records with no test prep course and math score > 70: 223
+------+--------------+---------------------------+------------+-----------------------+----------+-------------+-------------+
|gender|race/ethnicity|parental level of education|lunch       |test preparation course|math score|reading score|writing score|
+------+--------------+---------------------------+------------+-----------------------+----------+-------------+-------------+
|female|group B       |bachelor's degree          |standard    |none                   |72        |72           |74           |
|female|group B       |master's degree            |standard    |none                   |90        |95           |93           |
|male  |group C       |some college               |standard    |none                   |76        |78           |75           |
|female|group B       |associate's degree         |standard    |none                   |71        |83           |78           |
|male  |group C       |high school            